In [2]:
import os
import sys

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists("/kaggle/input")

if IS_KAGGLE:
    print("[environment] Kaggle GPU environment detected. running cloud setup...")
    if not os.path.exists("dental-segmentation"):
        !git clone https://github.com/ALZ-11/dental-segmentation.git
    
    sys.path.append("dental-segmentation/tf_baseline")
    base_dir = "/kaggle/input/datasets/pawanvalluri/dental-segmentation/dentalai-2"
else:
    print("[environment] local machine detected. running local configuration...")
    repo_root = os.path.abspath(os.path.join(os.getcwd(), "..",".."))
    sys.path.append(os.path.join(repo_root, "tf_baseline"))
    
    base_dir = os.path.join(repo_root, "dataset")
    if not os.path.exists(base_dir):
        base_dir = os.path.join(os.getcwd(), "dataset")

print(f" -> base dataset directory resolved to: {base_dir}")

[environment] Kaggle GPU environment detected. running cloud setup...
 -> base dataset directory resolved to: /kaggle/input/datasets/pawanvalluri/dental-segmentation/dentalai-2


In [3]:
from src.data_pipeline import compute_global_class_weights, get_dataset_pipeline

train_dir = os.path.join(base_dir, "train")
valid_dir = os.path.join(base_dir, "valid")
test_dir = os.path.join(base_dir, "test")

train_images = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith(".jpg")])
train_masks = sorted([os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.endswith("_mask.png")])

valid_images = sorted([os.path.join(valid_dir, f) for f in os.listdir(valid_dir) if f.endswith(".jpg")])
valid_masks = sorted([os.path.join(valid_dir, f) for f in os.listdir(valid_dir) if f.endswith("_mask.png")])

test_images = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith(".jpg")])
test_masks = sorted([os.path.join(test_dir, f) for f in os.listdir(test_dir) if f.endswith("_mask.png")])

print(f"[dataset discovery] train files: {len(train_images)}, valid files: {len(valid_images)}, test files: {len(test_images)}")

# compute global inverse frequency weights over all train files
weights_bg, weights_teeth = compute_global_class_weights(train_masks)
weights_list = [weights_bg, weights_teeth]

[dataset discovery] train files: 1991, valid files: 254, test files: 250
[preprocessing] computing global class weights over 1991 files...
 -> completed... background pixels: 10924220537, teeth pixels: 2068872988
 -> true global weights: [0: 0.5947, 1: 3.1401]


In [4]:
BATCH_SIZE = 16

# generate pre-fetched pipelines
train_ds = get_dataset_pipeline(train_images, train_masks, batch_size=BATCH_SIZE, is_training=True)
val_ds = get_dataset_pipeline(valid_images, valid_masks, batch_size=BATCH_SIZE, is_training=False)
test_ds = get_dataset_pipeline(test_images, test_masks, batch_size=BATCH_SIZE, is_training=False)

# check: pull 1 batch and verify shapes
for sample_images, sample_masks in train_ds.take(1):
    print(f"[pipeline check] image batch shape: {sample_images.shape} (expected: ({BATCH_SIZE}, 256, 256, 3))")
    print(f"[pipeline check] mask batch shape: {sample_masks.shape} (expected: ({BATCH_SIZE}, 256, 256, 2))")

I0000 00:00:1783814361.366734      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783814361.369765      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


[pipeline check] image batch shape: (16, 256, 256, 3) (expected: (16, 256, 256, 3))
[pipeline check] mask batch shape: (16, 256, 256, 2) (expected: (16, 256, 256, 2))


In [5]:
from src.model import unet_model
from src.loss import weighted_categorical_crossentropy
from src.metrics import ArgmaxMeanIoU

model = unet_model(input_shape=(256, 256, 3), num_classes=2)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=weighted_categorical_crossentropy(weights_list),
    metrics=["accuracy", ArgmaxMeanIoU(num_classes=2)]
)

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="best_model_optimized.keras",
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/20


2026-07-12 00:01:58.583043: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:01:58.866794: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:01:59.922914: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:02:00.232389: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:02:00.484303: E external/local_xla/xla/stream_

124/125 ━━━━━━━━━━━━━━━━━━━━ 0s 827ms/step - accuracy: 0.7349 - loss: 0.5409 - mean_io_u: 0.5247

2026-07-12 00:04:02.810778: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:04:03.104310: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:04:03.945824: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:04:04.253374: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-07-12 00:04:04.525451: E external/local_xla/xla/stream_

125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 988ms/step - accuracy: 0.7354 - loss: 0.5401 - mean_io_u: 0.5252
Epoch 1: val_loss improved from None to 0.51818, saving model to best_model_optimized.keras

Epoch 1: finished saving model to best_model_optimized.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 183s 1s/step - accuracy: 0.7969 - loss: 0.4356 - mean_io_u: 0.5967 - val_accuracy: 0.8421 - val_loss: 0.5182 - val_mean_io_u: 0.5751
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 826ms/step - accuracy: 0.8568 - loss: 0.3383 - mean_io_u: 0.6817
Epoch 2: val_loss improved from 0.51818 to 0.41343, saving model to best_model_optimized.keras

Epoch 2: finished saving model to best_model_optimized.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 122s 969ms/step - accuracy: 0.8575 - loss: 0.3319 - mean_io_u: 0.6814 - val_accuracy: 0.8021 - val_loss: 0.4134 - val_mean_io_u: 0.5739
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 828ms/step - accuracy: 0.8670 - loss: 0.3089 - mean_io_u: 0.6988
Epoch 3: val_loss improved from 0.41343 to 0.3087